## CA 4 - Large Language Models (Spring 2026)

- **Name: [Enter your name here]**
- **Student ID: [Enter your Student ID here]**

---
### Retrieval-Augmented Question Answering with Small Language Models

This notebook contains the step-by-step implementation of a **Retrieval-Augmented Generation (RAG)** pipeline for answering document-grounded multiple-choice questions. The goal is to evaluate whether retrieval can improve the performance of a small language model on questions generated from course-related PDF documents.

In this assignment, you will load and preprocess PDF files, split them into chunks, build both **sparse** and **dense** retrievers, and compare three question-answering pipelines:

1. A vanilla small language model without retrieval.
2. A RAG pipeline using a BM25 sparse retriever.
3. A RAG pipeline using a FAISS dense retriever.

Finally, you will evaluate the pipelines on a multiple-choice benchmark and compare their accuracy across different question difficulty levels.

In [ ]:
# Install the required packages

!pip install -q huggingface_hub langchain langchain-community pypdf 
!pip install -q sentence-transformers lancedb pyarrow tantivy langchain-text-splitters
!pip install -q faiss-cpu langchain-huggingface rank-bm25
!pip install -q transformers accelerate

In [ ]:
# the path to the pdfs 
from pathlib import Path
from typing import List

PDF_PATHS = [
    "data/10.pdf",
    "data/11.pdf",
    "data/13.pdf",
]

PDF_PATHS = [Path(p) for p in PDF_PATHS]

### Loading and preprocessing PDF documents (5 Points)

In this step, you must prepare the raw PDF documents for retrieval.

Your tasks are:

- Load each PDF file in `PDF_PATHS` using LangChain's `PyPDFLoader`.
- Convert each PDF into page-level LangChain `Document` objects.
- Remove the last two pages of each PDF because they mostly contain references or bibliography content.
- Add the source PDF filename to the metadata of every remaining page using the key `"source_file"`.
- Store all processed pages from all PDFs in the list `docs`.

At the end of this cell, `docs` should contain all usable page-level documents that will later be split into smaller chunks.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

docs = []

for pdf_path in PDF_PATHS:
    # TODO: Load the current PDF file using PyPDFLoader

    # TODO: Remove the last two pages of the loaded PDF

    # TODO: Add the source filename to the metadata of each remaining page

    # TODO: Add the processed pages to docs

print("Loaded pages after removing last two pages:", len(docs))

### Splitting documents into chunks (5 Points)

In this step, you must split the page-level documents into smaller text chunks.

Your tasks are:

- Create a `RecursiveCharacterTextSplitter`.
- Set an appropriate `chunk_size` so each chunk contains enough context for answering questions.
- Set a `chunk_overlap` so neighboring chunks share some text and important information is less likely to be cut off.
- Apply the text splitter to the loaded documents in `docs`.
- Store the resulting chunks in the variable `chunks`.

These chunks will be used as the searchable units for both sparse and dense retrieval.

**Question:**  
Why do we split long documents into smaller chunks before building a retrieval system?

**Answer:**  
[Write your answer here]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



text_splitter = RecursiveCharacterTextSplitter(
    # TODO: your code
)

chunks = # TODO: your code

print("Number of chunks:", len(chunks))

### Building sparse and dense retrievers (5 Points)

In this step, you must build two different retrievers over the document chunks.

Your tasks are:

- Build a **dense retriever** using sentence embeddings and FAISS.
- Use the embedding model `"sentence-transformers/all-MiniLM-L6-v2"` to convert chunks into vectors.
- Store the vector index using `FAISS.from_documents`.
- Convert the FAISS vector store into a retriever with `k = 5`.
- Build a **sparse retriever** using BM25.
- Use the same document chunks for BM25 retrieval.
- Set `k = 5` so both retrievers return the top 5 most relevant chunks.

**Question:**  
What is the main difference between sparse retrieval such as BM25 and dense retrieval using embeddings?

**Answer:**  
[Write your answer here]

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Create dense FAISS retriever

embedding_model = HuggingFaceEmbeddings(
    # TODO: Your code
)

vectorstore = # TODO: Use FAISS to create a vector store

dense_retriever = # TODO: Convert the FAISS vector store into a retriever with k = 5

print("Dense retriever is ready.")

In [ ]:
from langchain_community.retrievers import BM25Retriever


# Create BM25 sparse retriever

# TODO: Create a BM25 retriever from the document chunks

# TODO: Set the number of retrieved documents to 5
print("BM25 retriever is ready.")

In [ ]:
# Test both retrievers

query = "What is self attention?"

dense_results = dense_retriever.invoke(query)
bm25_results = bm25_retriever.invoke(query)

print("Dense results:")
for doc in dense_results:
    # print(doc.metadata)
    print(doc.page_content)
    print("-" * 80)

print("BM25 results:")
for doc in bm25_results:
    # print(doc.metadata)
    print(doc.page_content)
    print("-" * 80)

### Loading the language model (5 points)

In this step, you must load the language model that will answer the multiple-choice questions.

Your tasks are:

- Load `"Qwen/Qwen2.5-0.5B-Instruct"` and its tokenizer.
- Wrap the Hugging Face pipeline with LangChain's `HuggingFacePipeline`.

We use a small instruction-tuned model because the goal of this assignment is to test whether retrieval can improve the performance of a lightweight language model.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.llms import HuggingFacePipeline


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = # TODO

model = # TODO

pipe = # TODO

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, Any

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.runnables import RunnableLambda

In [ ]:
# loading test dataset
import json

with open("data/small_test_set.json", "r", encoding="utf-8") as f:
    small_questions = json.load(f)

with open("data/benchmark_set.json", "r", encoding="utf-8") as f:
    benchmark_questions = json.load(f)


print("Small set size:", len(small_questions))
print("Benchmark set size:", len(benchmark_questions))

### Creating the vanilla question-answering pipeline (20 points)

In this step, you must build the first question-answering pipeline: a **vanilla LLM pipeline** without retrieval.

Your tasks are:

- Define a Pydantic schema for the model output.
- The output must contain:
  - `answer`: one of `"A"`, `"B"`, `"C"`, or `"D"`.
  - `reason`: a short explanation for the selected answer.
    Example output:
    ```python
    {"answer": "B", "reason": "The context describes this method as using lexical overlap."}
    ```
- The pipeline input should be a dictionary with one key:
```python
{"question": "full multiple-choice question here"}
```
This means the `HumanMessagePromptTemplate` must contain the variable `{question}`.  
For example, the human prompt should be defined so that it receives the question text from the `"question"` field of the input dictionary.

- Create a JSON output parser using the schema.
- Build a prompt using LangChain message prompt templates.
- The prompt should ask the model to answer multiple-choice questions and return structured JSON.
- Create a LangChain pipeline that:
  - formats the prompt,
  - sends it to the language model,
  - parses the JSON output.

In this pipeline, you will use three types of prompt templates:

- `SystemMessagePromptTemplate`: defines the general behavior of the model, such as answering multiple-choice questions and returning JSON.
- `HumanMessagePromptTemplate`: represents the user input. In this assignment, it receives the full question through the `{question}` variable.
- `AIMessagePromptTemplate`: optionally gives the model a starting phrase for its answer, such as `"Here is the output in json:\n"`.

Large instruction-tuned models often follow structured-output instructions without needing an `AIMessagePromptTemplate`. However, smaller language models may produce extra text, explanations, or malformed JSON. Giving the model a short assistant-side prefix can help guide it toward the expected output format and reduce parsing errors.

This vanilla pipeline will be used as the baseline. Later, you will compare it with RAG pipelines that use retrieved context.

**Question:**  
Why is it useful to force the model to return structured JSON instead of free-form text?

**Answer:**  
[Write your answer here]

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate
)
from langchain_core.output_parsers import JsonOutputParser
from typing import Literal
from pydantic import BaseModel, Field



class Answer(BaseModel):
# TODO: Define a Pydantic class named Answer
# The class should contain:
# - answer: one of "A", "B", "C", or "D"
# - reason: a short string explaining the answer

json_parser = # TODO: Create a JSON parser using the Pydantic output schema


# TODO: Create a system prompt that tells the model:
# - it is answering multiple-choice questions
# - it must return valid JSON
# - the answer must be one of A, B, C, or D
# - the reason must be short
system_prompt = ... 

# TODO: Create a human prompt that receives the question as input
human_prompt = ...

# TODO: Optionally create an AI prompt prefix to encourage JSON output. 
# For small language models, this prompt is useful
ai_prompt = ...


prompt = ChatPromptTemplate.from_messages([
# TODO: Combine the system, human, and AI prompts into a ChatPromptTemplate

])


# TODO: Create the vanilla LangChain pipeline
# The pipeline should:
# 1. format the prompt
# 3. call the LLM
# 4. parse the JSON output

vanilla_chain = (
 ...
)

### Testing on a small dataset

You are provided with a small dataset to test your pipeline.
This step is useful because:

- It helps verify that the prompt is working correctly.
- It checks whether the model returns valid JSON in the expected format.
- It makes debugging easier before running the larger benchmark.
- It allows you to inspect the model's predicted answer and reason for each question.
- It helps you improve or simplify the prompt if the model produces invalid outputs.


In [ ]:
# a function for preparing the question

def format_mcq(example):
    return f"""
Question:
{example["question"]}

Choices:
A. {example["choices"]["A"]}
B. {example["choices"]["B"]}
C. {example["choices"]["C"]}
D. {example["choices"]["D"]}
""".strip()

In [ ]:
# tesing on one example

example = small_questions[0]

question = format_mcq(example)

output = vanilla_chain.invoke({
    "question": question
})

output

In [ ]:
# testing on a small dataset


from tqdm.auto import tqdm

small_outputs = []

for example in tqdm(small_questions, desc="Running vanilla model on small dataset"):
    question = format_mcq(example)

    try:
        model_output = vanilla_chain.invoke({
            "question": question
        })

        predicted_answer = model_output.get("answer", "INVALID")
        reason = model_output.get("reason", "")

    except Exception as e:
        model_output = None
        predicted_answer = "ERROR"
        reason = str(e)

    row = {
        "id": example["id"],
        "question": example["question"],
        "gold_answer": example["answer"],
        "predicted_answer": predicted_answer,
        "reason": reason,
        "is_correct": predicted_answer == example["answer"],
        "raw_output": model_output,
    }

    small_outputs.append(row)

    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print("Gold answer:", row["gold_answer"])
    print("Predicted answer:", row["predicted_answer"])
    print("Correct:", row["is_correct"])
    print("Reason:", row["reason"])

In [ ]:
import pandas as pd

small_results_df = pd.DataFrame(small_outputs)

accuracy = small_results_df["is_correct"].mean()

print("Small dataset accuracy:", accuracy)
print("Correct:", small_results_df["is_correct"].sum(), "/", len(small_results_df))

### Creating the sparse RAG pipeline with BM25 (20 Points)

In this step, you will build a **retrieval-augmented question-answering pipeline** using the BM25 sparse retriever.

Unlike the vanilla pipeline, this pipeline first retrieves relevant chunks from the PDF documents and then gives those chunks to the language model as additional context.

Your tasks are:

- Write a helper function to combine retrieved documents into one context string.
- Create a prompt that includes both:
  - the retrieved contexts,
  - the multiple-choice question.
- Use the BM25 retriever to retrieve relevant chunks for each question.
- Build a LangChain pipeline that adds the retrieved contexts before calling the model.
- Parse the final model output using the same JSON parser as before.

In this pipeline, `RunnablePassthrough` is used to keep the original input dictionary unchanged while adding a new key to it.

For example, the input to the chain has this format:

```python
{"question": "full multiple-choice question here"}
```

The sparse RAG chain should automatically add another field:

```python
{
    "question": "full multiple-choice question here",
    "contexts": "retrieved document chunks here"
}
```

This is useful because the original question is still needed by the prompt, but we also need to attach retrieved context before sending the input to the model.

`RunnablePassthrough.assign(...)` allows us to compute a new field, such as `contexts`, from the existing input while passing the rest of the input forward.

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    # TODO: Convert a list of retrieved documents into one context string
    # Hint: use the page_content field of each document
    ...


# TODO: Create a system prompt for sparse RAG
# The prompt should tell the model:
# - it is answering multiple-choice questions
# - it must return valid JSON
# - the answer must be one of A, B, C, or D
# - it should use the provided contexts when relevant
# - the reason must be short
sparse_system_prompt = ...


# TODO: Create a human prompt that receives both:
# - contexts
# - question
# Hint: the template should contain {contexts} and {question}
sparse_human_prompt = ...


# TODO: Optionally create an AI prompt prefix to encourage JSON output
sparse_ai_prompt = ...


sparse_prompt = ChatPromptTemplate.from_messages([
    # TODO: Combine the sparse system, human, and AI prompts
])


# TODO: Create the sparse RAG chain
# The pipeline should:
# 1. receive {"question": "..."}
# 2. retrieve relevant documents using bm25_retriever
# 3. format the retrieved documents into a context string using format_docs function
# 4. add the context string to the input using RunnablePassthrough.assign
# 5. format the prompt
# 6. call the LLM
# 7. parse the JSON output using the json parser you defined in the previous parts

sparse_rag_chain = (
    ...
)


In [ ]:
example = small_questions[0]

output = sparse_rag_chain.invoke({
    "question": format_mcq(example)
})

output

In [ ]:


sparse_small_outputs = []

for example in tqdm(small_questions, desc="Running sparse RAG on small dataset"):
    full_question = format_mcq(example)

    try:
        model_output = sparse_rag_chain.invoke({
            "question": full_question
        })

        predicted_answer = model_output.get("answer", "INVALID")
        reason = model_output.get("reason", "")

    except Exception as e:
        model_output = None
        predicted_answer = "ERROR"
        reason = str(e)

    row = {
        "id": example["id"],
        "question": example["question"],
        "gold_answer": example["answer"],
        "predicted_answer": predicted_answer,
        "reason": reason,
        "is_correct": predicted_answer == example["answer"],
        "raw_output": model_output,
    }

    sparse_small_outputs.append(row)

    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print("Gold answer:", row["gold_answer"])
    print("Predicted answer:", row["predicted_answer"])
    print("Correct:", row["is_correct"])
    print("Reason:", row["reason"])

In [ ]:
sparse_small_results_df = pd.DataFrame(sparse_small_outputs)
accuracy = sparse_small_results_df["is_correct"].mean()

print("Sparse RAG small dataset accuracy:", accuracy)
print("Correct:", sparse_small_results_df["is_correct"].sum(), "/", len(sparse_small_results_df))


### Creating the dense RAG pipeline with FAISS (20 Points)

In this step, you will build a second retrieval-augmented question-answering pipeline using the **dense FAISS retriever**.

This pipeline is similar to the sparse RAG pipeline, but instead of using BM25, it retrieves chunks based on embedding similarity. The question is converted into a vector, and FAISS searches for document chunks with similar vector representations.

Your tasks are:

- Create a prompt that includes both:
  - the retrieved contexts,
  - the multiple-choice question.
- Use the dense FAISS retriever to retrieve relevant chunks for each question.
- Add the retrieved contexts to the input using `RunnablePassthrough.assign`.
- Build a LangChain pipeline that sends the context-augmented prompt to the model.
- Parse the model output using the same JSON parser as before.

The input to the chain should still have the same simple format:

```python
{"question": "full multiple-choice question here"}
```

The dense RAG chain should internally retrieve relevant chunks and add them as `contexts` before calling the model.


In [ ]:
# TODO: Create a system prompt for dense RAG
# The prompt should tell the model:
# - it is answering multiple-choice questions
# - it must return valid JSON
# - the answer must be one of A, B, C, or D
# - it should use the provided contexts when relevant
# - the reason must be short
dense_system_prompt = ...


# TODO: Create a human prompt that receives both:
# - contexts
# - question
# Hint: the template should contain {contexts} and {question}
dense_human_prompt = ...


# TODO: Optionally create an AI prompt prefix to encourage JSON output
dense_ai_prompt = ...


dense_prompt = ChatPromptTemplate.from_messages([
    # TODO: Combine the dense system, human, and AI prompts
])


# TODO: Create the dense RAG chain
# The pipeline should:
# 1. receive {"question": "..."}
# 2. retrieve relevant documents using dense_retriever
# 3. format the retrieved documents into a context string
# 4. add the context string to the input using RunnablePassthrough.assign
# 5. format the prompt
# 6. call the LLM
# 7. parse the JSON output

dense_rag_chain = (
    ...
)

In [ ]:
example = small_questions[0]

output = dense_rag_chain.invoke({
    "question": format_mcq(example)
})

output

In [ ]:
dense_small_outputs = []

for example in tqdm(small_questions, desc="Running dense RAG on small dataset"):
    full_question = format_mcq(example)

    try:
        model_output = dense_rag_chain.invoke({
            "question": full_question
        })

        predicted_answer = model_output.get("answer", "INVALID")
        reason = model_output.get("reason", "")

    except Exception as e:
        model_output = None
        predicted_answer = "ERROR"
        reason = str(e)

    row = {
        "id": example["id"],
        "question": example["question"],
        "gold_answer": example["answer"],
        "predicted_answer": predicted_answer,
        "reason": reason,
        "is_correct": predicted_answer == example["answer"],
        "raw_output": model_output,
    }

    dense_small_outputs.append(row)

    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print("Gold answer:", row["gold_answer"])
    print("Predicted answer:", row["predicted_answer"])
    print("Correct:", row["is_correct"])
    print("Reason:", row["reason"])

In [ ]:
dense_small_results_df = pd.DataFrame(dense_small_outputs)


accuracy = dense_small_results_df["is_correct"].mean()

print("Dense RAG small dataset accuracy:", accuracy)
print("Correct:", dense_small_results_df["is_correct"].sum(), "/", len(dense_small_results_df))

### Final benchmarking and analysis (20 Points)

In this final step, you will benchmark the three question-answering pipelines on the larger MCQ dataset.

The three pipelines are:

- `vanilla_chain`: answers questions without retrieval.
- `sparse_rag_chain`: answers questions using BM25 sparse retrieval.
- `dense_rag_chain`: answers questions using FAISS dense retrieval.

You may run the evaluation on the full benchmark dataset, or sample a smaller subset of questions if generation is slow on your machine. For example, you can sample 10, 20, or 30 questions from `benchmark_questions`.

**Questions:**

1. Draw a bar chart of the final total accuracy for the three methods.
2. Analyze the results. Which method performed best?
3. Was using retrieval useful? Explain based on the accuracy results.
4. Was retrieval misleading on some questions? Give examples if possible.

**Answer:**

[Write your analysis here]

In [ ]:
import random
import pandas as pd
from tqdm.auto import tqdm

random.seed(42)

hard_questions = [
    q for q in benchmark_questions
    if q["difficulty"] == "hard"
]

# You can sample a smaller subset of questions if you are short on computing resources. 
# sampled_questions = random.sample(hard_questions, 20)
sampled_questions = benchmark_questions

print("Sampled questions:", len(sampled_questions))

In [ ]:
def run_chain_on_questions(chain, questions, method_name):
    rows = []

    for example in tqdm(questions, desc=f"Running {method_name}"):
        full_question = format_mcq(example)

        try:
            output = chain.invoke({
                "question": full_question
            })

            predicted_answer = output.get('answer', 'INVALID')
            reason = output.get('reason', '')

        except Exception as e:
            predicted_answer = "ERROR"
            reason = str(e)
            raw_output = None

        rows.append({
            "method": method_name,
            "id": example["id"],
            "difficulty": example["difficulty"],
            "question_type": example["question_type"],
            "gold_answer": example["answer"],
            "predicted_answer": predicted_answer,
            "is_correct": predicted_answer == example["answer"],
            "reason": reason,
        })

        print("=" * 80)
        print("Method:", method_name)
        print("ID:", example["id"])
        print("Difficulty:", example["difficulty"])
        print("Gold:", example["answer"])
        print("Predicted:", predicted_answer)
        print("Correct:", predicted_answer == example["answer"])
        print("Reason:", reason)

    return pd.DataFrame(rows)

In [ ]:
vanilla_results_df = run_chain_on_questions(
    vanilla_chain,
    sampled_questions,
    method_name="Vanilla"
)

sparse_results_df = run_chain_on_questions(
    sparse_rag_chain,
    sampled_questions,
    method_name="Sparse RAG"
)

dense_results_df = run_chain_on_questions(
    dense_rag_chain,
    sampled_questions,
    method_name="Dense RAG"
)

In [ ]:
all_results_df = pd.concat(
    [
        vanilla_results_df,
        sparse_results_df,
        dense_results_df,
    ],
    ignore_index=True,
)

all_results_df

In [ ]:
# TODO your code 